# BT4221 Group Project — Feature Engineering

## Objective
This notebook performs feature engineering for the Steam review recommendation classification task.

The purpose of this notebook is not merely to create more columns, but to transform the cleaned review dataset into a representation that is more predictive, interpretable, and aligned with the patterns discovered during exploratory data analysis (EDA).

In particular, the EDA suggested several important findings:

1. Many numeric variables, especially playtime and vote-related variables, are heavily right-skewed.
2. Some binary variables are informative, but not all of them contribute equally.
3. Game-level context matters, because recommendation behaviour differs noticeably across apps.
4. Review text is likely one of the strongest signal sources.
5. Some raw columns are not useful in their original form, but become valuable after transformation, interaction construction, or aggregation.

Therefore, the feature engineering strategy in this notebook is designed around five feature families:

- behavioural features
- engagement and vote features
- game-level contextual features
- text structure and lexical features
- LLM-assisted semantic text features using OpenAI API

## Motivations
This notebook is designed to support stronger performance under the project rubric by showing:

- clearly justified feature design choices instead of arbitrary transformations
- direct linkage between EDA findings and engineering decisions
- meaningful PySpark-based execution rather than only conceptual discussion
- interpretable and data-driven insights rather than black-box feature generation
- evidence of AI-assisted pipeline enhancement through OpenAI-generated structured text features

## Important methodological principle
To avoid data leakage, any aggregate feature that depends on the target distribution, such as app-level recommendation rate, must be computed on the training split only and then mapped into validation/test data.

This is important because leakage would inflate model performance and weaken the credibility of the project.

In [1]:
# ============================================
# 1. Environment setup
# ============================================

import os
import re
import json
import math
import ast
from typing import List, Dict

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler,
    Tokenizer,
    StopWordsRemover,
    CountVectorizer,
    IDF,
    HashingTF
)

from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("BT4221_Feature_Engineering") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [2]:
# ============================================
# 2. Load cleaned dataset
# ============================================

raw_path = r"C:\Justin Folder\UNI\projects\bt4221\BT4221-Project-Steam-Reviews\cleaned_steam_reviews.csv"

raw_df = spark.read.csv(
    raw_path,
    header=True,
    inferSchema=False,
    multiLine=True,
    escape='"'
)

print("Rows:", raw_df.count())
print("Columns:", len(raw_df.columns))
raw_df.printSchema()
raw_df.show(5, truncate=False)

Rows: 10777944
Columns: 22
root
 |-- app_id: string (nullable = true)
 |-- app_name: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- language: string (nullable = true)
 |-- review: string (nullable = true)
 |-- timestamp_created: string (nullable = true)
 |-- timestamp_updated: string (nullable = true)
 |-- recommended: string (nullable = true)
 |-- votes_helpful: string (nullable = true)
 |-- votes_funny: string (nullable = true)
 |-- weighted_vote_score: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- steam_purchase: string (nullable = true)
 |-- received_for_free: string (nullable = true)
 |-- written_during_early_access: string (nullable = true)
 |-- author_steamid: string (nullable = true)
 |-- author_num_games_owned: string (nullable = true)
 |-- author_num_reviews: string (nullable = true)
 |-- author_playtime_forever: string (nullable = true)
 |-- author_playtime_last_two_weeks: string (nullable = true)
 |-- author_playtime_at_r

In [3]:
expected_cols = [
    "app_id", "app_name", "review_id", "language", "review",
    "timestamp_created", "timestamp_updated", "recommended",
    "votes_helpful", "votes_funny", "weighted_vote_score",
    "comment_count", "steam_purchase", "received_for_free",
    "written_during_early_access", "author_steamid",
    "author_num_games_owned", "author_num_reviews",
    "author_playtime_forever", "author_playtime_last_two_weeks",
    "author_playtime_at_review", "author_last_played"
]

print("Actual columns:")
print(raw_df.columns)

missing_cols = [c for c in expected_cols if c not in raw_df.columns]
extra_cols = [c for c in raw_df.columns if c not in expected_cols]

print("Missing cols:", missing_cols)
print("Extra cols:", extra_cols)

Actual columns:
['app_id', 'app_name', 'review_id', 'language', 'review', 'timestamp_created', 'timestamp_updated', 'recommended', 'votes_helpful', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author_playtime_forever', 'author_playtime_last_two_weeks', 'author_playtime_at_review', 'author_last_played']
Missing cols: []
Extra cols: []


### Why audit before casting?

The dataset is first loaded entirely as strings to preserve the raw contents and avoid premature type conversion errors. This is especially important for review datasets, where multiline text, quotes, and malformed records can cause row shifts or invalid values.

Auditing the raw dataset first improves traceability and allows us to justify all later cleaning and casting decisions based on actual observed distributions rather than assumptions.

In [4]:
raw_df.select(
    F.count("*").alias("total_rows"),

    F.count(F.when(F.col("review").isNull(), 1)).alias("review_null"),
    F.count(F.when(F.trim(F.col("review")) == "", 1)).alias("review_blank"),

    F.count(F.when(F.col("app_name").isNull(), 1)).alias("app_name_null"),
    F.count(F.when(F.trim(F.col("app_name")) == "", 1)).alias("app_name_blank"),

    F.count(F.when(F.col("recommended").isNull(), 1)).alias("recommended_null"),
    F.count(F.when(~F.col("recommended").isin("0", "1"), 1)).alias("recommended_invalid")
).show(truncate=False)

print("Recommended distribution:")
raw_df.groupBy("recommended").count().orderBy(F.desc("count")).show(20, truncate=False)

+----------+-----------+------------+-------------+--------------+----------------+-------------------+
|total_rows|review_null|review_blank|app_name_null|app_name_blank|recommended_null|recommended_invalid|
+----------+-----------+------------+-------------+--------------+----------------+-------------------+
|10777944  |851825     |98          |515694       |939           |1130777         |404625             |
+----------+-----------+------------+-------------+--------------+----------------+-------------------+

Recommended distribution:
+------------------+-------+
|recommended       |count  |
+------------------+-------+
|1                 |8280319|
|NULL              |1130777|
|0                 |962223 |
|0.0               |153449 |
|1.0               |12063  |
|2.0               |5407   |
|0.523809552192688 |3930   |
|3.0               |2767   |
|4.0               |1810   |
|5.0               |1199   |
|6.0               |881    |
|0.4761904776096344|801    |
|0.545454561710357

In [5]:
df = raw_df.select(
    F.col("app_id").cast("string").alias("app_id"),
    F.col("app_name").cast("string").alias("app_name"),
    F.col("review_id").cast("string").alias("review_id"),
    F.col("language").cast("string").alias("language"),
    F.col("review").cast("string").alias("review"),
    F.expr("try_cast(timestamp_created as long)").alias("timestamp_created"),
    F.expr("try_cast(timestamp_updated as long)").alias("timestamp_updated"),
    F.expr("try_cast(recommended as int)").alias("recommended"),
    F.expr("try_cast(votes_helpful as double)").alias("votes_helpful"),
    F.expr("try_cast(votes_funny as double)").alias("votes_funny"),
    F.expr("try_cast(weighted_vote_score as double)").alias("weighted_vote_score"),
    F.expr("try_cast(comment_count as double)").alias("comment_count"),
    F.expr("try_cast(steam_purchase as int)").alias("steam_purchase"),
    F.expr("try_cast(received_for_free as int)").alias("received_for_free"),
    F.expr("try_cast(written_during_early_access as int)").alias("written_during_early_access"),
    F.col("author_steamid").cast("string").alias("author_steamid"),
    F.expr("try_cast(author_num_games_owned as double)").alias("author_num_games_owned"),
    F.expr("try_cast(author_num_reviews as double)").alias("author_num_reviews"),
    F.expr("try_cast(author_playtime_forever as double)").alias("author_playtime_forever"),
    F.expr("try_cast(author_playtime_last_two_weeks as double)").alias("author_playtime_last_two_weeks"),
    F.expr("try_cast(author_playtime_at_review as double)").alias("author_playtime_at_review"),
    F.expr("try_cast(author_last_played as double)").alias("author_last_played")
)

print("Rows after cast:", df.count())

Rows after cast: 10777944


In [6]:
invalid_condition = (
    F.col("review").isNull() |
    (F.trim(F.col("review")) == "") |
    F.col("app_name").isNull() |
    (F.trim(F.col("app_name")) == "") |
    F.col("recommended").isNull() |
    (~F.col("recommended").isin(0, 1))
)

print("Rows to be removed:", df.filter(invalid_condition).count())
print("Rows kept:", df.filter(~invalid_condition).count())

Rows to be removed: 1537645
Rows kept: 9240299


In [7]:
df = df.filter(~invalid_condition)

print("Final rows after basic filter:", df.count())
df.show(5, truncate=False)

Final rows after basic filter: 9240299
+------+-------------------------------------------+---------+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
print("Recommended distribution:")
df.groupBy("recommended").count().orderBy(F.desc("count")).show(20, truncate=False)

Recommended distribution:
+-----------+-------+
|recommended|count  |
+-----------+-------+
|1          |8278133|
|0          |962166 |
+-----------+-------+



## 4.1 Train-validation split before feature engineering

Before constructing the engineered feature set, we first split the cleaned dataset into training and validation sets. This is done early because some later feature engineering steps are leakage-sensitive.

In particular, app-level aggregate features and TF-IDF involve learning information from the data. If those transformations are fitted on the full dataset before splitting, information from the validation set would leak into the training pipeline and make downstream performance estimates overly optimistic.

To keep the pipeline methodologically sound, the split is performed first, and all learned or target-derived transformations are subsequently fitted only on the training data before being applied to the validation set.

In [9]:
train_df, valid_df = df.randomSplit([0.8, 0.2], seed=42)

print("Train rows:", train_df.count())
print("Valid rows:", valid_df.count())

print("Train target distribution")
train_df.groupBy("recommended").count().show()

print("Validation target distribution")
valid_df.groupBy("recommended").count().show()

Train rows: 7392534
Valid rows: 1847765
Train target distribution
+-----------+-------+
|recommended|  count|
+-----------+-------+
|          1|6622577|
|          0| 769957|
+-----------+-------+

Validation target distribution
+-----------+-------+
|recommended|  count|
+-----------+-------+
|          1|1655556|
|          0| 192209|
+-----------+-------+



## 4.2 Text structure features

The EDA suggested that review text is likely one of the most important sources of predictive signal. However, useful text information is not limited to vocabulary alone. Even before applying TF-IDF or semantic models, the structure of a review can already capture meaningful differences between positive and negative reviews.

For example, review length, punctuation intensity, and sentence structure may reflect:
- how much effort was put into the review
- whether the review is emotional or detailed
- whether it is a short impulsive reaction or a fuller evaluation

We therefore begin by engineering dense text-structure features that are interpretable and easy to explain in the report.

In [10]:
def add_text_structure_features(input_df):
    return input_df.withColumn("review_char_count", F.length("review")) \
        .withColumn("review_word_count", F.size(F.split(F.trim(F.col("review")), r"\s+"))) \
        .withColumn("review_sentence_count", F.size(F.split(F.col("review"), r"[.!?]+"))) \
        .withColumn("exclamation_count", F.length("review") - F.length(F.regexp_replace("review", "!", ""))) \
        .withColumn("question_count", F.length("review") - F.length(F.regexp_replace("review", r"\?", ""))) \
        .withColumn("digit_count", F.length("review") - F.length(F.regexp_replace("review", r"[0-9]", ""))) \
        .withColumn("has_exclamation", F.when(F.col("exclamation_count") > 0, 1).otherwise(0)) \
        .withColumn("has_question", F.when(F.col("question_count") > 0, 1).otherwise(0)) \
        .withColumn("long_review_flag", F.when(F.col("review_word_count") >= 100, 1).otherwise(0)) \
        .withColumn("very_short_review_flag", F.when(F.col("review_word_count") <= 10, 1).otherwise(0)) \
        .withColumn(
            "avg_sentence_length",
            F.when(F.col("review_sentence_count") > 0,
                   F.col("review_word_count") / F.col("review_sentence_count"))
             .otherwise(F.lit(0.0))
        )

train_df = add_text_structure_features(train_df)
valid_df = add_text_structure_features(valid_df)

train_df.select(
    "review_char_count",
    "review_word_count",
    "review_sentence_count",
    "exclamation_count",
    "question_count",
    "avg_sentence_length"
).show(5, truncate=False)

+-----------------+-----------------+---------------------+-----------------+--------------+-------------------+
|review_char_count|review_word_count|review_sentence_count|exclamation_count|question_count|avg_sentence_length|
+-----------------+-----------------+---------------------+-----------------+--------------+-------------------+
|9                |1                |2                    |0                |0             |0.5                |
|180              |33               |3                    |0                |0             |11.0               |
|118              |24               |4                    |0                |0             |6.0                |
|196              |35               |4                    |0                |0             |8.75               |
|55               |10               |2                    |0                |0             |5.0                |
+-----------------+-----------------+---------------------+-----------------+--------------+----

### Why these text structure features are justified

These features are justified because they convert the raw review into interpretable behavioural signals. In particular:

- `review_char_count` and `review_word_count` measure review length and effort
- `review_sentence_count` and `avg_sentence_length` measure how developed or fragmented the review is
- `exclamation_count` and `question_count` measure emphasis or frustration
- `long_review_flag` and `very_short_review_flag` help isolate extreme review styles

This is especially helpful for the rubric because these features are not black-box transformations. They can be clearly explained, visualised, and later discussed in relation to model performance and review behaviour.

## 4.3 Missingness indicators

Some metadata fields may be missing, and that missingness can itself carry signal. For example, the absence of playtime or vote-related metadata may reflect differences in review history, review age, or platform behaviour.

To preserve that information, we create missingness flags before filling nulls for downstream numeric transformations. This allows the modelling stage to distinguish between a true zero and an originally missing value.

In [11]:
missing_flag_cols = [
    "votes_helpful",
    "votes_funny",
    "weighted_vote_score",
    "comment_count",
    "author_num_games_owned",
    "author_num_reviews",
    "author_playtime_forever",
    "author_playtime_last_two_weeks",
    "author_playtime_at_review",
    "author_last_played"
]

def add_missing_flags(input_df, cols):
    out = input_df
    for c in cols:
        if c in out.columns:
            out = out.withColumn(f"{c}_missing", F.when(F.col(c).isNull(), 1).otherwise(0))
    return out

train_df = add_missing_flags(train_df, missing_flag_cols)
valid_df = add_missing_flags(valid_df, missing_flag_cols)

## 4.3 Log-transform skewed numeric features

The EDA indicated that many numeric variables are strongly right-skewed. This is particularly common for online review data, where most observations are small but a few reviews or users have extremely large values.

This matters because raw skewed variables can distort modelling by allowing extreme observations to dominate the scale. Rather than discarding such columns, a better approach is to retain them but transform them into a more stable form.

We therefore apply a `log1p` transformation to key engagement and playtime-related features.

In [12]:
log_numeric_cols = [
    "votes_helpful",
    "votes_funny",
    "comment_count",
    "author_num_games_owned",
    "author_num_reviews",
    "author_playtime_forever",
    "author_playtime_last_two_weeks",
    "author_playtime_at_review"
]

def add_log_features(input_df, cols):
    out = input_df
    for c in cols:
        if c in out.columns:
            out = out.withColumn(f"log1p_{c}", F.log1p(F.coalesce(F.col(c), F.lit(0))))
    return out

train_df = add_log_features(train_df, log_numeric_cols)
valid_df = add_log_features(valid_df, log_numeric_cols)

train_df.select(
    "votes_helpful", "log1p_votes_helpful",
    "author_playtime_forever", "log1p_author_playtime_forever"
).show(5, truncate=False)

+-------------+-------------------+-----------------------+-----------------------------+
|votes_helpful|log1p_votes_helpful|author_playtime_forever|log1p_author_playtime_forever|
+-------------+-------------------+-----------------------+-----------------------------+
|19.0         |2.995732273553991  |4291.0                 |8.36450810375059             |
|32.0         |3.4965075614664802 |79.0                   |4.382026634673881            |
|0.0          |0.0                |11842.0                |9.379492254721495            |
|0.0          |0.0                |29943.0                |10.307084249584264           |
|5.0          |1.791759469228055  |1605.0                 |7.381501894506707            |
+-------------+-------------------+-----------------------+-----------------------------+
only showing top 5 rows


### Why the log transformation is justified

This transformation is directly supported by the EDA. It improves the representation of skewed variables by:

- reducing the influence of extreme outliers
- preserving rank ordering
- compressing very large values into a more model-friendly scale
- keeping potentially useful variables instead of dropping them

This is a stronger feature engineering decision than either leaving the variables untouched or removing them too early.

## 4.4 Behavioural ratio features

Raw counts are useful, but they do not always capture behaviour as well as ratios do. Two users may have similar totals while behaving very differently.

To address this, we engineer several ratio-based features that reflect reviewer behaviour more directly. These help move the feature space from simple quantity to more meaningful behavioural patterns.

In [13]:
def add_ratio_features(input_df):
    return input_df.withColumn(
            "funny_to_helpful_ratio",
            (F.coalesce(F.col("votes_funny"), F.lit(0)) + 1) /
            (F.coalesce(F.col("votes_helpful"), F.lit(0)) + 1)
        ).withColumn(
            "reviews_to_games_ratio",
            (F.coalesce(F.col("author_num_reviews"), F.lit(0)) + 1) /
            (F.coalesce(F.col("author_num_games_owned"), F.lit(0)) + 1)
        ).withColumn(
            "recent_to_total_playtime_ratio",
            (F.coalesce(F.col("author_playtime_last_two_weeks"), F.lit(0)) + 1) /
            (F.coalesce(F.col("author_playtime_forever"), F.lit(0)) + 1)
        ).withColumn(
            "words_per_playtime",
            (F.coalesce(F.col("review_word_count"), F.lit(0)) + 1) /
            (F.coalesce(F.col("author_playtime_at_review"), F.lit(0)) + 1)
        ).withColumn(
            "helpful_per_word",
            (F.coalesce(F.col("votes_helpful"), F.lit(0)) + 1) /
            (F.coalesce(F.col("review_word_count"), F.lit(0)) + 1)
        ).withColumn(
            "comment_per_word",
            (F.coalesce(F.col("comment_count"), F.lit(0)) + 1) /
            (F.coalesce(F.col("review_word_count"), F.lit(0)) + 1)
        ).withColumn(
            "playtime_per_game_owned",
            (F.coalesce(F.col("author_playtime_forever"), F.lit(0)) + 1) /
            (F.coalesce(F.col("author_num_games_owned"), F.lit(0)) + 1)
        )

train_df = add_ratio_features(train_df)
valid_df = add_ratio_features(valid_df)

train_df.select(
    "funny_to_helpful_ratio",
    "reviews_to_games_ratio",
    "recent_to_total_playtime_ratio",
    "words_per_playtime",
    "helpful_per_word",
    "comment_per_word",
    "playtime_per_game_owned"
).show(5, truncate=False)

+----------------------+----------------------+------------------------------+---------------------+--------------------+--------------------+-----------------------+
|funny_to_helpful_ratio|reviews_to_games_ratio|recent_to_total_playtime_ratio|words_per_playtime   |helpful_per_word    |comment_per_word    |playtime_per_game_owned|
+----------------------+----------------------+------------------------------+---------------------+--------------------+--------------------+-----------------------+
|0.15                  |0.143646408839779     |2.3299161230195712E-4         |0.0028735632183908046|10.0                |1.0                 |23.712707182320443     |
|0.030303030303030304  |0.04326923076923077   |0.0125                        |0.425                |0.9705882352941176  |0.08823529411764706 |0.19230769230769232    |
|1.0                   |0.05925925925925926   |0.10081904922739171           |0.00249003984063745  |0.04                |0.04                |87.72592592592592      

### Why these ratio features are justified

These features provide additional behavioural interpretation beyond raw counts:

- `funny_to_helpful_ratio` distinguishes comedic or sarcastic reviews from useful ones
- `reviews_to_games_ratio` measures how actively a user reviews relative to how many games they own
- `recent_to_total_playtime_ratio` captures whether the review is based on recent engagement
- `words_per_playtime` approximates whether review detail is proportionate to actual play exposure

These features are useful because they reflect reviewer behaviour and review credibility in a more nuanced way than standalone counts.

## 4.5 Binary context features

The dataset contains several binary context variables such as:
- `steam_purchase`
- `received_for_free`
- `written_during_early_access`

The EDA suggested that some of these may be stronger than others. However, weaker univariate predictors should not be removed prematurely, because they may still contribute jointly with other features in multivariate models.

For this reason, we retain these binary variables as part of the engineered feature set.

In [14]:
binary_cols = ["steam_purchase", "received_for_free", "written_during_early_access"]

for c in binary_cols:
    if c in train_df.columns:
        train_df = train_df.withColumn(c, F.coalesce(F.col(c), F.lit(0)))
        valid_df = valid_df.withColumn(c, F.coalesce(F.col(c), F.lit(0)))

train_df.select(*[c for c in binary_cols if c in train_df.columns]).show(5, truncate=False)

+--------------+-----------------+---------------------------+
|steam_purchase|received_for_free|written_during_early_access|
+--------------+-----------------+---------------------------+
|1             |0                |0                          |
|1             |0                |1                          |
|1             |0                |0                          |
|0             |1                |0                          |
|1             |0                |0                          |
+--------------+-----------------+---------------------------+
only showing top 5 rows


### Why keeping weaker binary variables is still justified

A variable that looks weak on its own is not necessarily useless in the final model. For example, `received_for_free` may have limited standalone predictive power, but it can still interact with text sentiment, game type, or review length.

Retaining these variables at this stage is therefore more defensible than dropping them based only on univariate inspection.

## 4.6 Leakage-safe app-level contextual features

The EDA suggested that `app_name` is highly informative, which means game-level context is important. However, app-level statistics can easily cause leakage if they are computed using the full dataset, especially when they involve the target variable.

To avoid this, all app-level aggregates are computed on the training set only, then joined back into both the training and validation sets. This ensures that validation rows do not influence the aggregate values used during training.

In [15]:
train_app_stats = train_df.groupBy("app_name").agg(
    F.count("*").alias("app_review_count"),
    F.avg("recommended").alias("app_recommend_rate"),
    F.avg("review_word_count").alias("app_avg_review_word_count"),
    F.avg(F.coalesce(F.col("votes_helpful"), F.lit(0))).alias("app_avg_votes_helpful")
)

global_defaults = train_app_stats.agg(
    F.avg("app_review_count").alias("global_app_review_count"),
    F.avg("app_recommend_rate").alias("global_app_recommend_rate"),
    F.avg("app_avg_review_word_count").alias("global_app_avg_review_word_count"),
    F.avg("app_avg_votes_helpful").alias("global_app_avg_votes_helpful")
).collect()[0]

train_df = train_df.join(train_app_stats, on="app_name", how="left")
valid_df = valid_df.join(train_app_stats, on="app_name", how="left")

train_df = train_df.fillna({
    "app_review_count": global_defaults["global_app_review_count"],
    "app_recommend_rate": global_defaults["global_app_recommend_rate"],
    "app_avg_review_word_count": global_defaults["global_app_avg_review_word_count"],
    "app_avg_votes_helpful": global_defaults["global_app_avg_votes_helpful"]
})

valid_df = valid_df.fillna({
    "app_review_count": global_defaults["global_app_review_count"],
    "app_recommend_rate": global_defaults["global_app_recommend_rate"],
    "app_avg_review_word_count": global_defaults["global_app_avg_review_word_count"],
    "app_avg_votes_helpful": global_defaults["global_app_avg_votes_helpful"]
})

train_df.select(
    "app_name",
    "app_review_count",
    "app_recommend_rate",
    "app_avg_review_word_count",
    "app_avg_votes_helpful"
).show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+------------------+-------------------------+---------------------+
|app_name                                                                                                                                                                                                                                                                                          |app_review_count|app_recommend_rate|app_avg_review_word_count|app_avg_votes_helpful|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Why app-level features are justified

This is one of the most important feature families in the notebook. The EDA suggested that recommendation rates vary across games, which means the model should be given access to game-level context.

The engineered app features capture:
- how widely reviewed a game is
- how positively it is received overall
- how long reviews tend to be for that game
- how much helpful engagement that game’s reviews receive

This is more informative than simply passing the raw `app_name` into the model, because it translates the categorical label into interpretable contextual signals.

## 4.9 Temporal and edit-history features

The dataset contains timestamp metadata that can be used to capture when the review was created, whether it was edited later, and how recent the player’s last-played activity was relative to the review.

These features are useful because they may reflect:
- review timing in the game lifecycle
- whether the user returned to revise the review
- whether the review was written close to the player’s most recent engagement

In [16]:
def add_time_features(input_df):
    out = input_df.withColumn("created_ts", F.from_unixtime(F.col("timestamp_created")).cast("timestamp")) \
        .withColumn("updated_ts", F.from_unixtime(F.col("timestamp_updated")).cast("timestamp")) \
        .withColumn("last_played_ts", F.from_unixtime(F.col("author_last_played")).cast("timestamp")) \
        .withColumn("created_year", F.year("created_ts")) \
        .withColumn("created_month", F.month("created_ts")) \
        .withColumn("created_dayofweek", F.dayofweek("created_ts")) \
        .withColumn("is_weekend_review", F.when(F.col("created_dayofweek").isin(1, 7), 1).otherwise(0)) \
        .withColumn(
            "review_edit_gap_seconds",
            F.when(
                F.col("timestamp_updated").isNotNull() & F.col("timestamp_created").isNotNull(),
                F.col("timestamp_updated") - F.col("timestamp_created")
            ).otherwise(None)
        ).withColumn(
            "review_edited_flag",
            F.when(
                F.col("timestamp_updated").isNotNull() &
                F.col("timestamp_created").isNotNull() &
                (F.col("timestamp_updated") > F.col("timestamp_created")),
                1
            ).otherwise(0)
        ).withColumn(
            "time_since_last_played_seconds",
            F.when(
                F.col("timestamp_created").isNotNull() &
                F.col("author_last_played").isNotNull(),
                F.col("timestamp_created") - F.col("author_last_played")
            ).otherwise(None)
        )

    out = out.withColumn("review_edit_gap_seconds", F.coalesce(F.col("review_edit_gap_seconds"), F.lit(0))) \
             .withColumn("time_since_last_played_seconds", F.coalesce(F.col("time_since_last_played_seconds"), F.lit(0)))

    return out

train_df = add_time_features(train_df)
valid_df = add_time_features(valid_df)

train_df.select(
    "created_year",
    "created_month",
    "created_dayofweek",
    "is_weekend_review",
    "review_edit_gap_seconds",
    "review_edited_flag",
    "time_since_last_played_seconds"
).show(5, truncate=False)

+------------+-------------+-----------------+-----------------+-----------------------+------------------+------------------------------+
|created_year|created_month|created_dayofweek|is_weekend_review|review_edit_gap_seconds|review_edited_flag|time_since_last_played_seconds|
+------------+-------------+-----------------+-----------------+-----------------------+------------------+------------------------------+
|2015        |11           |5                |0                |0                      |0                 |-9.5013385E7                  |
|2018        |1            |4                |0                |0                      |0                 |2790.0                        |
|2020        |3            |4                |0                |0                      |0                 |-2.6691809E7                  |
|2018        |2            |5                |0                |20684510               |1                 |-5.052985E7                   |
|2016        |10           

### Why temporal features are justified

Temporal features are relatively cheap to create and can still be useful because they capture:
- the timing of a review in the game lifecycle
- possible weekday versus weekend behavioural differences
- longer-run temporal shifts in review patterns

Even if their predictive contribution is modest, they are interpretable and worth including as part of a well-rounded feature set.

## 4.11 Leakage-safe TF-IDF text features

Sparse lexical text features are retained because the EDA suggested that review text is highly informative. While the text-structure features above capture how a review is written, TF-IDF captures what is being said.

To keep the transformation leakage-safe, the IDF component is fitted on the training set only, and the fitted model is then applied to both training and validation data.

In [17]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF

tokenizer = Tokenizer(inputCol="review", outputCol="tokens_raw")
remover = StopWordsRemover(inputCol="tokens_raw", outputCol="tokens_clean")
hashing_tf = HashingTF(inputCol="tokens_clean", outputCol="tf_features", numFeatures=5000)
idf = IDF(inputCol="tf_features", outputCol="tfidf_features")

train_df = tokenizer.transform(train_df)
valid_df = tokenizer.transform(valid_df)

train_df = remover.transform(train_df)
valid_df = remover.transform(valid_df)

train_df = hashing_tf.transform(train_df)
valid_df = hashing_tf.transform(valid_df)

idf_model = idf.fit(train_df)

train_df = idf_model.transform(train_df)
valid_df = idf_model.transform(valid_df)

train_df.select("tokens_clean", "tfidf_features").show(3, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|tokens_clean                                                                                                                 |tfidf_features                                                                                                                                                                                                                                                                                                                                                         |
+-----------------------

### Why TF-IDF is justified

TF-IDF complements the earlier dense text features in an important way:

- dense text features capture **how** a review is written
- TF-IDF captures **what** a review is saying

This gives the model a richer text representation and is more defensible than relying on review length alone. It also aligns well with the project objective of extracting meaningful predictive information from text-heavy review data.

## 4.12 Final engineered dense feature list

To make the feature engineering output reproducible and easy to hand over for downstream modelling, we explicitly define the dense engineered feature set produced in this notebook.

This is separate from the sparse lexical representation in `tfidf_features`.

In [18]:
dense_feature_cols = [
    "review_char_count",
    "review_word_count",
    "review_sentence_count",
    "exclamation_count",
    "question_count",
    "digit_count",
    "has_exclamation",
    "has_question",
    "long_review_flag",
    "very_short_review_flag",
    "avg_sentence_length",

    "log1p_votes_helpful",
    "log1p_votes_funny",
    "log1p_comment_count",
    "log1p_author_num_games_owned",
    "log1p_author_num_reviews",
    "log1p_author_playtime_forever",
    "log1p_author_playtime_last_two_weeks",
    "log1p_author_playtime_at_review",

    "weighted_vote_score_filled",

    "funny_to_helpful_ratio",
    "reviews_to_games_ratio",
    "recent_to_total_playtime_ratio",
    "words_per_playtime",
    "helpful_per_word",
    "comment_per_word",
    "playtime_per_game_owned",

    "steam_purchase",
    "received_for_free",
    "written_during_early_access",

    "created_year",
    "created_month",
    "created_dayofweek",
    "is_weekend_review",
    "review_edit_gap_seconds",
    "review_edited_flag",
    "time_since_last_played_seconds",

    "app_review_count",
    "app_recommend_rate",
    "app_avg_review_word_count",
    "app_avg_votes_helpful",

    "votes_helpful_missing",
    "votes_funny_missing",
    "weighted_vote_score_missing",
    "comment_count_missing",
    "author_num_games_owned_missing",
    "author_num_reviews_missing",
    "author_playtime_forever_missing",
    "author_playtime_last_two_weeks_missing",
    "author_playtime_at_review_missing",
    "author_last_played_missing"
]

dense_feature_cols = [c for c in dense_feature_cols if c in train_df.columns]

print("Dense feature count:", len(dense_feature_cols))
print(dense_feature_cols)

Dense feature count: 50
['review_char_count', 'review_word_count', 'review_sentence_count', 'exclamation_count', 'question_count', 'digit_count', 'has_exclamation', 'has_question', 'long_review_flag', 'very_short_review_flag', 'avg_sentence_length', 'log1p_votes_helpful', 'log1p_votes_funny', 'log1p_comment_count', 'log1p_author_num_games_owned', 'log1p_author_num_reviews', 'log1p_author_playtime_forever', 'log1p_author_playtime_last_two_weeks', 'log1p_author_playtime_at_review', 'funny_to_helpful_ratio', 'reviews_to_games_ratio', 'recent_to_total_playtime_ratio', 'words_per_playtime', 'helpful_per_word', 'comment_per_word', 'playtime_per_game_owned', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'created_year', 'created_month', 'created_dayofweek', 'is_weekend_review', 'review_edit_gap_seconds', 'review_edited_flag', 'time_since_last_played_seconds', 'app_review_count', 'app_recommend_rate', 'app_avg_review_word_count', 'app_avg_votes_helpful', 'votes_helpful_m

## 4.13 Output of the feature engineering stage

At the end of this section, the feature engineering stage produces:

- `train_df`, containing the training split with engineered dense features and sparse TF-IDF text features
- `valid_df`, containing the validation split transformed using the same leakage-safe feature logic
- `dense_feature_cols`, which explicitly lists the engineered dense features for downstream modelling
- `tfidf_features`, which stores the sparse lexical text representation

This provides a clean and modelling-ready handoff for the downstream training and evaluation stage.

In [21]:
print("Train rows after FE:", train_df.count())
print("Valid rows after FE:", valid_df.count())

train_df.select("recommended", *dense_feature_cols[:10]).show(5, truncate=False)
valid_df.select("recommended", *dense_feature_cols[:10]).show(5, truncate=False)

Train rows after FE: 7392534
Valid rows after FE: 1847765
+-----------+-----------------+-----------------+---------------------+-----------------+--------------+-----------+---------------+------------+----------------+----------------------+
|recommended|review_char_count|review_word_count|review_sentence_count|exclamation_count|question_count|digit_count|has_exclamation|has_question|long_review_flag|very_short_review_flag|
+-----------+-----------------+-----------------+---------------------+-----------------+--------------+-----------+---------------+------------+----------------+----------------------+
|1          |9                |1                |2                    |0                |0             |0          |0              |0           |0               |1                     |
|0          |180              |33               |3                    |0                |0             |0          |0              |0           |0               |0                     |
|1          

## X.X OpenAI-assisted semantic feature engineering

Classical text features such as TF-IDF are useful because they capture word-level patterns, but they do not always capture higher-level meaning reliably. Two reviews may describe the same issue using very different vocabulary, for example poor performance, repetitive gameplay, or strong value for money.

To address this, we add a small semantic enrichment step using the OpenAI API. The goal is not to generate free-form summaries, but to convert review text into **structured semantic features** that can be used like ordinary model inputs.

We use the model to return a fixed schema for each review, including:
- sentiment label
- sentiment strength
- main topic
- recommendation clarity
- specificity level
- whether the review mentions bugs
- whether it mentions performance
- whether it mentions value for money
- whether it mentions replayability

This approach is more defensible than vague LLM prompting because the output is structured, reproducible, and directly connected to the downstream training pipeline.

### Why this step is justified

This step is justified for three reasons.

First, it is aligned with the EDA. Earlier analysis suggested that review text is one of the strongest feature sources, so it is reasonable to invest in richer text representations.

Second, it fits the rubric well. The project brief explicitly allows and encourages meaningful AI-assisted pipeline design, especially when the outputs are structured and actually influence downstream execution.

Third, semantic features fill a gap left by TF-IDF. TF-IDF captures lexical information, but semantic labels can better capture higher-level concepts such as whether a review is mainly about bugs, performance, value, or gameplay, even when different words are used.

### Practical note on scale

The full dataset is extremely large, so applying the API to every row would be slow and expensive. For this reason, the semantic enrichment step should be run on a manageable subset.

Reasonable choices include:
- a random sample
- the training split only
- a subset of the most informative reviews
- a smaller balanced sample used for modelling

The purpose of this section is to demonstrate a valid and structured LLM-assisted feature engineering workflow, not to exhaustively process the entire raw dataset.

### API setup

We initialise the OpenAI client using an environment variable so that secrets are not hardcoded into the notebook.

In [19]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is not set in the environment.")

client = OpenAI(api_key=api_key)
print("API key loaded")

API key loaded


### Step 1: sample a manageable subset directly from Spark

Because the full dataset is too large for API enrichment, we first select a manageable subset of valid reviews. This should ideally be drawn from the training data only if the modelling pipeline is already split.

In [20]:
llm_input_df = df.select("review_id", "review", "recommended") \
    .filter(F.col("review").isNotNull()) \
    .filter(F.trim(F.col("review")) != "") \
    .limit(1000)

print("Rows selected for LLM enrichment:", llm_input_df.count())
llm_input_df.show(5, truncate=False)

Rows selected for LLM enrichment: 1000
+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+
|review_id|rev